In [217]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

In [218]:
RETRIEVAL_DIR = Path("../output/retrieval")
DATA_DIR = Path("../data/gold_data")

METHOD = "hybrid_instruct" # "bm25" # # "bm25_rerank" # "dense" # "dense_instruct" # "dense_instruct_rerank" # "dense_rerank" # hybrid # "hybrid_instruct" # "hybrid_instruct_rerank" # "hybrid_rerank"

RESULTS_DIR = Path(f"../results/retrieval/{METHOD}")
RESULTS_DIR.mkdir(parents=False, exist_ok=True)

In [219]:
# Evaluate either with or w/o graphics
NO_GRAPHICS = True

if NO_GRAPHICS:
    gold_processed_df = pd.read_csv(DATA_DIR / "gold_standard_processed_no_graphics.csv")
else:
    gold_processed_df = pd.read_csv(DATA_DIR / "gold_standard_processed.csv")

In [220]:
print(f"Gold before filtering NA: {len(gold_processed_df)}")
gold_processed_df = gold_processed_df.dropna(subset=["page"])
print(f"Gold after filtering NA: {len(gold_processed_df)}")

Gold before filtering NA: 617
Gold after filtering NA: 547


In [221]:
# Display types of each gold page, e.g. {("Allianz_2022", 79.0): {"Table"}}
display_types_by_page = gold_processed_df.groupby(["report_id", "page"])["display_type"].agg(set).to_dict()
display_types_by_page

{('Allianz_2022', 79.0): {'Table'},
 ('Daimler_2020', 63.0): {'Table'},
 ('Fresenius SE_2019', 126.0): {'Table'},
 ('acuity brands inc_2022', 99.0): {'Table'},
 ('acuity brands inc_2022', 101.0): {'Table', 'Table/Text', 'Text/Table'},
 ('addtech_2022', 73.0): {'Table'},
 ('aixtron_2020', 51.0): {'Text'},
 ('allfunds group_2021', 125.0): {'Table'},
 ('allfunds group_2021', 142.0): {'Table'},
 ('americold realty inc_2022', 53.0): {'Table'},
 ('apollo commercial real estate fina_2019', 9.0): {'Table'},
 ('arteria networks corp_2019', 10.0): {'Table'},
 ('autoneum holding_2019', 27.0): {'Table'},
 ('balanced commercial property trust_2020', 34.0): {'Table'},
 ('bekaert (d) sa_2022', 276.0): {'Table'},
 ('blackberry ltd_2022', 21.0): {'Table'},
 ('blackberry ltd_2022', 22.0): {'Table'},
 ('blackline inc_2021', 21.0): {'Table'},
 ('cabot corp_2018', 22.0): {'Table'},
 ('cardinal energy ltd_2021', 19.0): {'Table'},
 ('century communities inc_2019', 11.0): {'Table'},
 ('chase corp_2020', 32.0)

In [222]:
gold_pages_by_report = gold_processed_df.groupby("report_id")["page"].agg(set).to_dict()
gold_pages_by_report

{'Allianz_2022': {79.0},
 'Daimler_2020': {63.0},
 'Fresenius SE_2019': {126.0},
 'acuity brands inc_2022': {99.0, 101.0},
 'addtech_2022': {73.0},
 'aixtron_2020': {51.0},
 'allfunds group_2021': {125.0, 142.0},
 'americold realty inc_2022': {53.0},
 'apollo commercial real estate fina_2019': {9.0},
 'arteria networks corp_2019': {10.0},
 'autoneum holding_2019': {27.0},
 'balanced commercial property trust_2020': {34.0},
 'bekaert (d) sa_2022': {276.0},
 'blackberry ltd_2022': {21.0, 22.0},
 'blackline inc_2021': {21.0},
 'cabot corp_2018': {22.0},
 'cardinal energy ltd_2021': {19.0},
 'century communities inc_2019': {11.0},
 'chase corp_2020': {32.0},
 'comerica inc_2019': {44.0},
 'cubesmart reit_2021': {19.0},
 'cytokinetics inc_2022': {24.0},
 'dampskibsselskabet norden_2019': {63.0},
 'dksh holding_2021': {73.0, 74.0},
 'evolution mining ltd_2020': {44.0},
 'fabege_2022': {45.0},
 'fastly inc_2018': {43.0},
 'freni brembo_2018': {131.0},
 'fuji soft inc_2022': {80.0, 81.0},
 'ge

In [223]:
retrieval_files = list((RETRIEVAL_DIR/METHOD).rglob("*.json"))

results = []

for retrieval_file in retrieval_files:
    # if retrieval_file.stem.split("_")[0] != "amano corp":
    #     continue
    with open(retrieval_file, "r", encoding="utf-8") as f:
        rdata = json.load(f)
    
    
    retrieval_method = rdata.get("method")
    
    report_id = rdata.get("report_id")
    if report_id not in gold_pages_by_report:
        continue
    
    retrieved_results = rdata.get("items", [])
    retrieved_pages = set()
    
    for item in retrieved_results:
        metadata = item.get("metadata", {})
        page_numbers = metadata.get("page_no")
        retrieved_pages.add(page_numbers)
    
    gold_pages = gold_pages_by_report[report_id]
    
    # Evaluate retrieval results
    true_positives = len(retrieved_pages & gold_pages)
    false_positives = len(retrieved_pages - gold_pages)
    false_negatives = len(gold_pages - retrieved_pages)
        
    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
    recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    missed_pages = sorted(gold_pages - retrieved_pages)

    missed_display_types = []
    for page in missed_pages:
        for display_type in display_types_by_page[(report_id, page)]:
            missed_display_types.append(str(display_type))
    
    results.append({
        "report_id": report_id,
        "precision": precision,
        "recall": recall,
        "f1_score": f1_score,
        "gold_pages": list(gold_pages),
        "retrieved_pages": list(retrieved_pages),
        "missed_pages": missed_pages,
        "annotated_display_type_missed_pages": missed_display_types,
    })
 
print(f"Retrieval Method: {retrieval_method}")
df = pd.DataFrame(results)


if NO_GRAPHICS:
    df.to_csv(f"{RESULTS_DIR}/{METHOD}_retrieval_eval_no_graphics.csv", index=False)
else:
    df.to_csv(f"{RESULTS_DIR}/{METHOD}_retrieval_eval.csv", index=False)
df

Retrieval Method: hybrid


,report_id,precision,recall,f1_score,gold_pages,retrieved_pages,missed_pages,annotated_display_type_missed_pages
0,acuity brands inc_2022,0.4,1.00,0.571429,"[99.0, 101.0]","[99, 100, 101, 102, 28]",[],[]
1,addtech_2022,0.2,1.00,0.333333,[73.0],"[69, 71, 73, 74, 21]",[],[]
2,aixtron_2020,0.2,1.00,0.333333,[51.0],"[42, 47, 81, 50, 51]",[],[]
3,allfunds group_2021,0.4,1.00,0.571429,"[125.0, 142.0]","[128, 142, 125, 126, 127]",[],[]
4,Allianz_2022,0.2,1.00,0.333333,[79.0],"[79, 49, 82, 51, 93]",[],[]
...,...,...,...,...,...,...,...,...
59,the simply good foods company_2018,0.2,1.00,0.333333,[91.0],"[1, 108, 115, 90, 91]",[],[]
60,toshiba tec corp_2022,0.2,1.00,0.333333,[31.0],"[34, 15, 27, 30, 31]",[],[]
61,uniper_2019,0.6,0.75,0.666667,"[16.0, 66.0, 67.0, 15.0]","[65, 66, 67, 16, 18]",[15.0],[Table]
62,varta ag_2021,0.2,1.00,0.333333,[39.0],"[35, 39, 40, 24, 27]",[],[]


In [224]:
df["precision"].mean(), df["recall"].mean(), df["f1_score"].mean()

(np.float64(0.234375), np.float64(0.95703125), np.float64(0.367373511904762))